In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer, AutoTokenizer
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Optional
from tqdm import tqdm
import random
# from models.gpt2_test import PrunableGPT2LMHeadModel as CircuitDiscoveryGPT2, GPT2LMHeadModel, PruningConfig

from models.gemma3 import Gemma3ForCausalLM
from dataset.ioi_t import IOIDataset, load_or_generate_ioi_data, run_evaluation

import torch
import torch.nn as nn
from tqdm import tqdm
from models.l0 import HardConcreteGate

import torch
import torch.nn as nn
from tqdm import tqdm
from utils import disable_dropout, analyze_and_finalize_circuit

# ==============================================================================
# PRUNING CONFIGURATION
# ==============================================================================
from dataclasses import dataclass


In [ ]:
PRUNING_FACTOR = 5
@dataclass
class PruningConfig:
    init_value: float = 1.0
    sparsity_warmup_steps: int = 0

    # --- Fine-grained pruning (existing) ---
    # Attention Head Pruning
    prune_attention_heads: bool = True
    lambda_attention_heads: float = 3 * PRUNING_FACTOR # 0.027 * PRUNING_FACTOR

    # MLP neuron pruning
    prune_mlp_hidden: bool = True
    lambda_mlp_hidden: float = 10 * PRUNING_FACTOR
    prune_mlp_output: bool = True
    lambda_mlp_output: float = 4 * PRUNING_FACTOR
    
    
    prune_attention_neurons: bool = True
    lambda_attention_neurons: float = 3 * PRUNING_FACTOR
    
    prune_embedding: bool = False
    lambda_embedding: float = 1 * PRUNING_FACTOR
    
    # Prune entire attention blocks
    prune_attention_blocks: bool = True
    lambda_attention_blocks: float = 0.2 * PRUNING_FACTOR
    
    # Prune entire MLP blocks
    prune_mlp_blocks: bool = True
    lambda_mlp_blocks: float = 0.1 * PRUNING_FACTOR
    
    # Prune entire transformer layers
    prune_full_layers: bool = False
    lambda_full_layers: float = 0.000000005 * PRUNING_FACTOR

import time
# ==============================================================================
# MAIN EXECUTION FOR IOI TASK
# ==============================================================================
if __name__ == '__main__':
    # --- Configuration ---
    MODEL_NAME = 'google/gemma-3-4b-pt'
    NUM_EPOCHS = 500
    LEARNING_RATE = 3e-3
    BATCH_SIZE = 1
    MAX_SEQ_LEN = 32
    ACCURACY_BUDGET = 0.05  # Allow 5% accuracy drop from baseline
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    

    pruning_config = PruningConfig()
    
    # --- Model and Tokenizer Setup ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    
    full_model = Gemma3ForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()
    # full_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
    for param in full_model.parameters(): param.requires_grad = False

    # ----- Disable all built-in dropout layers in the circuit model ---
    # print("\n--- Disabling all built-in dropout layers in the circuit model ---")
    # disable_dropout(circuit_model)
    # # -----------------------------------------------------------------
    
    # # --- Freeze the base model and unfreeze only the gates ---
    # print("Freezing base model weights and unfreezing gate parameters...")
    # total_params = 0
    # trainable_params = 0
    # for name, param in circuit_model.named_parameters():
    #     total_params += param.numel()
    #     if 'gate' not in name:
    #         param.requires_grad = False
    #     else:
    #         print(f"  Unfreezing for training: {name}")
    #         param.requires_grad = True
    #         trainable_params += param.numel()
            
    # print(f"\nTotal parameters: {total_params}")
    # print(f"Trainable gate parameters: {trainable_params} ({trainable_params/total_params*100:.4f}%)")

    # print("\nVerifying trainable parameters:")
    # for name, param in circuit_model.named_parameters():
    #     if param.requires_grad:
    #         print(f"  TRAINABLE: {name} - shape: {param.shape}")

    # # Double-check optimizer
    # print(f"\nOptimizer is training {len(optimizer.param_groups[0]['params'])} parameter tensors")

    # --- Dataset Setup ---
    print("\nSetting up IOI dataset...")
    # Load from disk
    train_data = load_or_generate_ioi_data(split="train", num_samples=400)  # Limit samples for efficiency
    # train_data += load_or_generate_ioi_data(split="test_10", num_samples=200)
    val_data = load_or_generate_ioi_data(split="validation", num_samples=200)
    test_data = load_or_generate_ioi_data(split="test")

    # Create dataset objects
    train_dataset = IOIDataset(train_data, tokenizer, max_length=MAX_SEQ_LEN)
    val_dataset = IOIDataset(val_data, tokenizer, max_length=MAX_SEQ_LEN)
    test_dataset = IOIDataset(test_data, tokenizer, max_length=MAX_SEQ_LEN)

    # Create dataloaders
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    test_dataloader = DataLoader(test_dataset, batch_size=128)

    # --- Baseline Evaluation ---
    print("\n--- Baseline evaluation on full model ---")
    baseline_results = run_evaluation(
        model_to_eval=full_model, 
        model_name="Baseline Full Model", 
        full_model_for_faithfulness=None, 
        dataloader=test_dataloader, 
        device=DEVICE, 
        tokenizer=tokenizer
    )
    # dummy_base_results = {'accuracy': 0.8423, 'logit_diff': 2.345, 'kl_div': 0.0, 'exact_match': 0.0}
    # baseline_results = dummy_base_results
    base_accuracy = baseline_results.get("accuracy", 0.0)
    base_logit_diff = baseline_results.get("logit_diff", 0.0)
    
    # --- Initial Circuit Model Evaluation ---
    # print("\n--- Initial evaluation of the Circuit Discovery Model ---")
    # circuit_model.eval()
    # initial_results = run_evaluation(
    #     model_to_eval=circuit_model, 
    #     model_name="Initial Circuit Model", 
    #     full_model_for_faithfulness=full_model, 
    #     dataloader=val_dataloader, 
    #     device=DEVICE, 
    #     tokenizer=tokenizer
    # )
    # initial_accuracy = initial_results.get("accuracy", 0.0)
    # initial_logit_diff = initial_results.get("logit_diff", 0.0)

    # # --- Training ---
    # # The optimizer will now only see the parameters that require gradients (the gates)
    # gate_params = [p for p in circuit_model.parameters() if p.requires_grad]
    # optimizer = AdamW(gate_params, lr=LEARNING_RATE)
    
    
    
    print(f"\n--- Starting training to find 'Indirect Object Identification' circuit ---")
    print(f"Target: Maintain accuracy within {ACCURACY_BUDGET*100}% of baseline ({base_accuracy:.4f})")

    # circuit_model.train()
    # total_steps = 0
    # time_start = time.time()
    # for epoch in range(NUM_EPOCHS):
    #     epoch_loss = 0
    #     epoch_kl_loss = 0
    #     epoch_sparsity_loss = 0
        
    #     for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
    #         optimizer.zero_grad()
            
    #         # Move batch to device
    #         for key, val in batch.items():
    #             if isinstance(val, torch.Tensor):
    #                 batch[key] = val.to(DEVICE)
            
    #         # Forward pass through circuit model with corrupted inputs
    #         circuit_outputs = circuit_model(
    #             input_ids=batch['input_ids'],
    #             corrupted_input_ids=batch['corrupted_input_ids'],
    #             attention_mask=batch['attention_mask']
    #         )
            
    #         # Get target outputs from full model
    #         with torch.no_grad():
    #             target_outputs = full_model(
    #                 input_ids=batch['input_ids'],
    #                 attention_mask=batch['attention_mask']
    #             )
            
    #         # Calculate loss at the prediction positions
    #         batch_size = circuit_outputs.logits.size(0)
    #         total_kl = 0
            
    #         for i in range(batch_size):
    #             t_start = batch['T_Start'][i].item()-1
    #             t_end = batch['T_End'][i].item()-1
                
    #             # if(len(batch['target_tokens'][i])>1):
    #             #     print(len(batch['target_tokens'][i]), batch['target_tokens'][i], tokenizer.decode(batch['target_tokens'][i]))
                
    #             # Get valid sequence length to avoid padding
    #             valid_length = batch['attention_mask'][i].sum().item()
    #             end_pos = min(t_end, valid_length)
                
    #             if t_start < end_pos:
    #                 circuit_logits = circuit_outputs.logits[i, t_start:end_pos]
    #                 target_logits = target_outputs.logits[i, t_start:end_pos]
                    
    #                 # KL divergence loss
    #                 kl = F.kl_div(
    #                     F.log_softmax(circuit_logits, dim=-1),
    #                     F.log_softmax(target_logits, dim=-1),
    #                     reduction='batchmean',
    #                     log_target=True
    #                 )
    #                 total_kl += kl
            
    #         kl_loss = total_kl / batch_size
    #         sparsity_loss = circuit_model.get_sparsity_loss(step=total_steps)['total_sparsity']
            
    #         # Total loss
    #         loss = kl_loss + sparsity_loss
    #         loss.backward()
    #         optimizer.step()
            
    #         # Track losses
    #         epoch_loss += loss.item()
    #         epoch_kl_loss += kl_loss.item()
    #         epoch_sparsity_loss += sparsity_loss.item()
    #         total_steps += 1circuit_model.train()
    # total_steps = 0
    # time_start = time.time()
    # for epoch in range(NUM_EPOCHS):
    #     epoch_loss = 0
    #     epoch_kl_loss = 0
    #     epoch_sparsity_loss = 0
        
    #     for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
    #         optimizer.zero_grad()
            
    #         # Move batch to device
    #         for key, val in batch.items():
    #             if isinstance(val, torch.Tensor):
    #                 batch[key] = val.to(DEVICE)
            
    #         # Forward pass through circuit model with corrupted inputs
    #         circuit_outputs = circuit_model(
    #             input_ids=batch['input_ids'],
    #             corrupted_input_ids=batch['corrupted_input_ids'],
    #             attention_mask=batch['attention_mask']
    #         )
            
    #         # Get target outputs from full model
    #         with torch.no_grad():
    #             target_outputs = full_model(
    #                 input_ids=batch['input_ids'],
    #                 attention_mask=batch['attention_mask']
    #             )
            
    #         # Calculate loss at the prediction positions
    #         batch_size = circuit_outputs.logits.size(0)
    #         total_kl = 0
            
    #         for i in range(batch_size):
    #             t_start = batch['T_Start'][i].item()-1
    #             t_end = batch['T_End'][i].item()-1
                
    #             # if(len(batch['target_tokens'][i])>1):
    #             #     print(len(batch['target_tokens'][i]), batch['target_tokens'][i], tokenizer.decode(batch['target_tokens'][i]))
                
    #             # Get valid sequence length to avoid padding
    #             valid_length = batch['attention_mask'][i].sum().item()
    #             end_pos = min(t_end, valid_length)
                
    #             if t_start < end_pos:
    #                 circuit_logits = circuit_outputs.logits[i, t_start:end_pos]
    #                 target_logits = target_outputs.logits[i, t_start:end_pos]
                    
    #                 # KL divergence loss
    #                 kl = F.kl_div(
    #                     F.log_softmax(circuit_logits, dim=-1),
    #                     F.log_softmax(target_logits, dim=-1),
    #                     reduction='batchmean',
    #                     log_target=True
    #                 )
    #                 total_kl += kl
            
    #         kl_loss = total_kl / batch_size
    #         sparsity_loss = circuit_model.get_sparsity_loss(step=total_steps)['total_sparsity']
            
    #         # Total loss
    #         loss = kl_loss + sparsity_loss
    #         loss.backward()
    #         optimizer.step()
            
    #         # Track losses
    #         epoch_loss += loss.item()
    #         epoch_kl_loss += kl_loss.item()
    #         epoch_sparsity_loss += sparsity_loss.item()
    #         total_steps += 1
    #     time_end = time.time()
        
    #     epoch_loss /= len(train_dataloader)
    #     epoch_kl_loss /= len(train_dataloader)
    #     epoch_sparsity_loss /= len(train_dataloader)

    #     print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} - Loss: {epoch_loss:.4f} | KL Loss: {epoch_kl_loss:.4f} | Sparsity Loss: {epoch_sparsity_loss:.4f} | Time: {time_end - time_start:.2f}s \n")
        
    #     # circuit_model.train()
    
    # print("\n--- Analyzing and finalizing circuit ---")
    # # pruning_config.prune_full_layers = True  # Enable full layer pruning for final evaluation
    # # circuit_model.set_pruning_config(pruning_config)
    # # --- Final Evaluation on Test Set ---
    # # analyze_and_finalize_circuit(circuit_model)

    # # --- Final Analysis and Pruning ---
    # print("\n--- Analyzing and finalizing circuit ---")
    # analyze_and_finalize_circuit(circuit_model)
    
    # # --- Final Evaluation on Test Set ---
    # print("\n--- Final evaluation on test set ---")
    # circuit_model.eval()
    # final_results = run_evaluation(
    #     model_to_eval=circuit_model, 
    #     model_name="Final Pruned Circuit (Optimal Thresholds)", 
    #     full_model_for_faithfulness=full_model, 
    #     dataloader=test_dataloader, 
    #     device=DEVICE, 
    #     tokenizer=tokenizer
    # )
    
    # # --- Summary ---
    # print("\n" + "="*60)
    # print("FINAL SUMMARY - IOI Circuit Discovery")
    # print("="*60)
    # print(f"Baseline Accuracy: {base_accuracy:.4f}")
    # print(f"Baseline Logit Diff: {base_logit_diff:.4f}")
    # print(f"Final Circuit Accuracy: {final_results['accuracy']:.4f} (drop: {base_accuracy - final_results['accuracy']:.4f})")
    # print(f"Final Circuit Logit Diff: {final_results['logit_diff']:.4f}")
    # print(f"Final KL Divergence: {final_results['kl_div']:.4f}")
    # print(f"Exact Match Rate: {final_results['exact_match']:.4f}")
    
    # # Get sparsity statistics
    # sparsity_stats = circuit_model.get_sparsity_loss(step=total_steps)
    # print(f"\nSparsity Statistics:")
    # for key, value in sparsity_stats.items():
    #     if key != 'total_sparsity':
    #         print(f"  - {key}: {value:.4f}")
    # print("="*60)
    #     time_end = time.time()
        
    #     epoch_loss /= len(train_dataloader)
    #     epoch_kl_loss /= len(train_dataloader)
    #     epoch_sparsity_loss /= len(train_dataloader)

    #     print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} - Loss: {epoch_loss:.4f} | KL Loss: {epoch_kl_loss:.4f} | Sparsity Loss: {epoch_sparsity_loss:.4f} | Time: {time_end - time_start:.2f}s \n")
        
    #     # circuit_model.train()
    
    # print("\n--- Analyzing and finalizing circuit ---")
    # # pruning_config.prune_full_layers = True  # Enable full layer pruning for final evaluation
    # # circuit_model.set_pruning_config(pruning_config)
    # # --- Final Evaluation on Test Set ---
    # # analyze_and_finalize_circuit(circuit_model)

    # # --- Final Analysis and Pruning ---
    # print("\n--- Analyzing and finalizing circuit ---")
    # analyze_and_finalize_circuit(circuit_model)
    
    # # --- Final Evaluation on Test Set ---
    # print("\n--- Final evaluation on test set ---")
    # circuit_model.eval()
    # final_results = run_evaluation(
    #     model_to_eval=circuit_model, 
    #     model_name="Final Pruned Circuit (Optimal Thresholds)", 
    #     full_model_for_faithfulness=full_model, 
    #     dataloader=test_dataloader, 
    #     device=DEVICE, 
    #     tokenizer=tokenizer
    # )
    
    # # --- Summary ---
    # print("\n" + "="*60)
    # print("FINAL SUMMARY - IOI Circuit Discovery")
    # print("="*60)
    # print(f"Baseline Accuracy: {base_accuracy:.4f}")
    # print(f"Baseline Logit Diff: {base_logit_diff:.4f}")
    # print(f"Final Circuit Accuracy: {final_results['accuracy']:.4f} (drop: {base_accuracy - final_results['accuracy']:.4f})")
    # print(f"Final Circuit Logit Diff: {final_results['logit_diff']:.4f}")
    # print(f"Final KL Divergence: {final_results['kl_div']:.4f}")
    # print(f"Exact Match Rate: {final_results['exact_match']:.4f}")
    
    # # Get sparsity statistics
    # sparsity_stats = circuit_model.get_sparsity_loss(step=total_steps)
    # print(f"\nSparsity Statistics:")
    # for key, value in sparsity_stats.items():
    #     if key != 'total_sparsity':
    #         print(f"  - {key}: {value:.4f}")
    # print("="*60)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Setting up IOI dataset...
Attempting to load dataset from: /u/amo-d1/grad/mha361/work/circuits/data/datasets/ioi
Successfully loaded train split with 200 samples
Attempting to load dataset from: /u/amo-d1/grad/mha361/work/circuits/data/datasets/ioi
Successfully loaded validation split with 200 samples
Attempting to load dataset from: /u/amo-d1/grad/mha361/work/circuits/data/datasets/ioi
Successfully loaded test split with 36084 samples
Processed 200 valid samples from 200 total
Processed 200 valid samples from 200 total
Processed 36084 valid samples from 36084 total

--- Baseline evaluation on full model ---

  EVALUATING: Baseline Full Model


Evaluating Baseline Full Model:   0%|          | 0/282 [00:00<?, ?it/s]/homes/mha361/work/circuits/dataset/ioi_t.py:244: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "target_tokens": torch.tensor(target_tokens, dtype=torch.long),
/homes/mha361/work/circuits/dataset/ioi_t.py:245: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "distractor_tokens": torch.tensor(distractor_tokens, dtype=torch.long),


OutOfMemoryError: CUDA out of memory. Tried to allocate 8.00 GiB. GPU 0 has a total capacity of 23.57 GiB of which 5.96 GiB is free. Including non-PyTorch memory, this process has 17.58 GiB memory in use. Of the allocated memory 16.67 GiB is allocated by PyTorch, and 633.58 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)